# Notebook 01 — CFR Ingestion Tracer (DC 5260)

Walk through the smallest end-to-end CFR ingestion: fetch §4.71a XML from eCFR, locate the text block for **DC 5260 (Leg, limitation of flexion of)**, send it through an LLM extractor with a Pydantic schema, gate the result with a deterministic validator, and write the resulting nodes to Neo4j.

**Prereqs**

- `make up` — Neo4j running
- `OPENAI_API_KEY` set in `.env`

## 1. Fetch the section XML

`fetch_section_xml` caches its response under `data/ecfr_cache/`. Re-running this cell is free after the first call.

In [1]:
from va_agent.ingestion.ecfr_client import fetch_section_xml, content_hash

xml_text, cache_path = fetch_section_xml(section='4.71a')
print(f'Fetched {len(xml_text):,} bytes of XML. Cached at:')
print(f'  {cache_path}')
print(f'Content hash: {content_hash(xml_text)[:16]}...')

Fetched 123,337 bytes of XML. Cached at:
  /Users/leightontidwell/Documents/personal-repos/va-disability-agent/data/ecfr_cache/2025-01-01-t38-p4-s4.71a.xml
Content hash: 6570a5b7ad4b6111...


## 2. Locate DC 5260's text block

§4.71a is encoded as a `<TABLE>` of `<TR>` rows. `extract_dc_text` scans rows whose first `<TD>` begins with the requested code and collects subsequent rows until the next DC header. The result is a clean text block the LLM can extract from.

In [2]:
from va_agent.ingestion.extraction import extract_dc_text

raw_text = extract_dc_text(xml_text, '5260')
print(raw_text)

5260 Leg, limitation of flexion of:
Flexion limited to 15° — 30%
Flexion limited to 30° — 20%
Flexion limited to 45° — 10%
Flexion limited to 60° — 0%


## 3. LLM extraction → Pydantic

`OpenAIExtractor` uses LangChain's `with_structured_output(DiagnosticCodeExtraction)` so the model can only return values that conform to our Pydantic schema. The model is constrained but can still hallucinate values within those constraints — the validator (next step) catches what slips through.

In [3]:
from va_agent.ingestion.extraction import OpenAIExtractor

extractor = OpenAIExtractor()
extraction = extractor.extract(raw_text)

print(f'DC {extraction.code}: {extraction.title}')
print(f'  body_system: {extraction.body_system}, section: {extraction.section}')
print(f'  rating_levels: {len(extraction.rating_levels)}')
for level in sorted(extraction.rating_levels, key=lambda lv: lv.percent):
    print(f'    {level.percent:>3}%  {level.criteria[0].text}')
    for m in level.criteria[0].measurements:
        print(f'         ↳ {m.name} {m.operator} {m.value} {m.unit}')

DC 5260: Leg, limitation of flexion of
  body_system: musculoskeletal, section: 4.71a
  rating_levels: 4
      0%  Flexion limited to 60°
         ↳ flexion <= 60.0 degrees
     10%  Flexion limited to 45°
         ↳ flexion <= 45.0 degrees
     20%  Flexion limited to 30°
         ↳ flexion <= 30.0 degrees
     30%  Flexion limited to 15°
         ↳ flexion <= 15.0 degrees


## 4. Deterministic validation

`validate_diagnostic_code` checks structural invariants:

- code is 4 digits
- body_system is in the closed set
- every rating_level.percent is in the canonical VA set (0, 10, 20, …, 100)
- no duplicate percents
- every measurement has a unit
- cross-references look like `DC NNNN` or `§X.YZ`

Anything failing here goes to `data/review_queue.jsonl` instead of the graph.

In [4]:
from va_agent.ingestion.validation import validate_diagnostic_code

result = validate_diagnostic_code(extraction)
print(f'ok: {result.ok}')
if result.errors:
    print('errors:', result.errors)
if result.warnings:
    print('warnings:', result.warnings)

ok: True


## 5. Write to graph (whole pipeline)

`ingest_diagnostic_code` orchestrates the whole thing: fetch → locate → extract → validate → write. The IngestionReport tells you what landed and what was diverted to the review queue.

In [5]:
from va_agent.graph.driver import GraphDriver
from va_agent.ingestion.cfr import ingest_diagnostic_code

with GraphDriver.from_env() as driver:
    # Clean state for a deterministic demo.
    driver.cfr_write('MATCH (dc:CFR:DiagnosticCode {code: $code}) DETACH DELETE dc', code='5260')

    report = ingest_diagnostic_code(section='4.71a', dc_code='5260', driver=driver)

    print('attempted:', report.dc_codes_attempted)
    print('written:  ', report.dc_codes_written)
    print('failed:   ', report.dc_codes_failed)
    print('warnings: ', report.warnings)

    print()
    print('Graph state for DC 5260:')
    rows = driver.cfr_read(
        '''MATCH (dc:CFR:DiagnosticCode {code: $code})-[:HAS_RATING]->(rl:RatingLevel)-[:REQUIRES]->(c:Criterion)
           OPTIONAL MATCH (c)-[:HAS_MEASUREMENT]->(m:Measurement)
           RETURN rl.percent AS percent, c.text AS criterion, collect(m{.name, .value, .unit}) AS measurements
           ORDER BY percent''',
        code='5260',
    )
    for r in rows:
        ms = r['measurements']
        m_str = '' if not ms or ms == [{}] else f'  measurements={ms}'
        print(f'  {r["percent"]:>3}%  {r["criterion"]}{m_str}')

attempted: ['5260']
written:   ['5260']
failed:    []
warnings:  []

Graph state for DC 5260:
    0%  Flexion limited to 60°  measurements=[{'name': 'flexion', 'unit': 'degrees', 'value': 60.0}]
   10%  Flexion limited to 45°  measurements=[{'name': 'flexion', 'unit': 'degrees', 'value': 45.0}]
   20%  Flexion limited to 30°  measurements=[{'name': 'flexion', 'unit': 'degrees', 'value': 30.0}]
   30%  Flexion limited to 15°  measurements=[{'name': 'flexion', 'unit': 'degrees', 'value': 15.0}]


## 6. Inspect in Neo4j Browser

Open <http://localhost:7474> and try:

```cypher
MATCH (dc:CFR:DiagnosticCode {code: '5260'})-[:HAS_RATING]->(rl)
OPTIONAL MATCH (rl)-[:REQUIRES]->(c)-[:HAS_MEASUREMENT]->(m)
RETURN dc, rl, c, m
```

You should see one DC node, four RatingLevel nodes (0/10/20/30%), four Criterion nodes, and four Measurement nodes.